## Dataset Overview:

This project uses a synthetic dataset created for learning and experimentation purposes. The dataset simulates a typical retail/business scenario containing numerical and categorical features along with intentionally introduced missing values.

The main objective of using this dataset is not to build a high-performing predictive model, but to explore and compare different missing value handling techniques and understand their impact on downstream machine learning performance.

In [190]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [191]:
df = pd.read_csv('../data/synthetic_dataset.csv')

In [192]:
df.head()

,Category,Price,Rating,Stock,Discount
0,NaN,5548.0,1.870322,NaN,0.0
1,NaN,3045.0,4.757798,NaN,38.0
2,NaN,4004.0,NaN,In Stock,0.0
3,NaN,4808.0,1.492085,NaN,33.0
4,NaN,1817.0,NaN,Out of Stock,23.0


In [193]:
df.shape

(4362, 5)

In [194]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4362 entries, 0 to 4361
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Category  1614 non-null   str    
 1   Price     4188 non-null   float64
 2   Rating    2312 non-null   float64
 3   Stock     3010 non-null   str    
 4   Discount  3970 non-null   float64
dtypes: float64(3), str(2)
memory usage: 170.5 KB


In [195]:
df['Category'].value_counts(ascending=True)

Category
B    378
A    403
D    408
C    425
Name: count, dtype: int64

In [196]:
df['Stock'].value_counts(ascending=True)

Stock
Out of Stock    1497
In Stock        1513
Name: count, dtype: int64

In [197]:
df.isna().sum()

Category    2748
Price        174
Rating      2050
Stock       1352
Discount     392
dtype: int64

In [198]:
df.describe()

,Price,Rating,Discount
count,4188.000000,2312.000000,3970.000000
mean,5016.970630,3.038293,24.516625
std,2839.984813,1.143074,14.347164
min,102.000000,1.000366,0.000000
25%,2628.250000,2.069490,12.000000
50%,4996.500000,3.082060,25.000000
75%,7418.000000,4.008620,37.000000
max,9999.000000,4.997818,49.000000


In [199]:
num_cols = ['Price', 'Rating', 'Discount']
cat_cols = ['Category', 'Stock']

In [200]:
df[num_cols].corr()

,Price,Rating,Discount
Price,1.000000,0.021099,0.009300
Rating,0.021099,1.000000,0.015028
Discount,0.009300,0.015028,1.000000


## Handling Missing Values

In [201]:
for col_name in df.columns:
    missing_percent = df[col_name].isnull().mean() * 100
    print(f'Missing Values Percentage in {col_name}: {missing_percent}%')

Missing Values Percentage in Category: 62.99862448418156%
Missing Values Percentage in Price: 3.988995873452544%
Missing Values Percentage in Rating: 46.99679046309033%
Missing Values Percentage in Stock: 30.994956441999083%
Missing Values Percentage in Discount: 8.986703347088492%


### Median + Mode Imputations on copy1 of df

#### Median Imputation on Numerical Columns

In [202]:
num_cols = ['Rating', 'Discount']
cat_cols = ['Category', 'Stock']

In [203]:
df_dup1 = df.copy()

In [204]:
for col in num_cols:
    df_dup1[col] = df_dup1[col].fillna(df_dup1[col].median())
    
df_dup1[num_cols].isnull().sum()

Rating      0
Discount    0
dtype: int64

#### Mode Imputation on Categorical Columns

In [205]:
for col in cat_cols:
    df_dup1[col] = df_dup1[col].fillna(df_dup1[col].mode()[0])
    
df_dup1[cat_cols].isnull().sum()

Category    0
Stock       0
dtype: int64

### Mean + Mode Imputations on copy2 of df

### Mean Imputation on Numerical Columns

In [206]:
df_dup2 = df.copy()

In [207]:
for col in num_cols:
    df_dup2[col] = df_dup2[col].fillna(df_dup2[col].mean())

df_dup1[num_cols].isnull().sum()

Rating      0
Discount    0
dtype: int64

### Mode Imputation on Categorical Columns

In [208]:
for col in cat_cols:
    df_dup2[col] = df_dup2[col].fillna(df_dup2[col].mode()[0])
    
df_dup2[cat_cols].isnull().sum()

Category    0
Stock       0
dtype: int64

### Mean + Mode Imputations + Missing Indicator on copy3 of df

In [209]:
df_dup3 = df.copy()

In [210]:
for col in num_cols:
    df_dup3[col + "_missing"] = df_dup3[col].isnull().astype(int)

    df_dup3[col] = df_dup3[col].fillna(df_dup3[col].mean())

df_dup3.head()

,Category,Price,Rating,Stock,Discount,Rating_missing,Discount_missing
0,NaN,5548.0,1.870322,NaN,0.0,0,0
1,NaN,3045.0,4.757798,NaN,38.0,0,0
2,NaN,4004.0,3.038293,In Stock,0.0,1,0
3,NaN,4808.0,1.492085,NaN,33.0,0,0
4,NaN,1817.0,3.038293,Out of Stock,23.0,1,0


In [211]:
for col in cat_cols:
    df_dup3[col + "_missing"] = df_dup3[col].isnull().astype(int)

    df_dup3[col] = df_dup3[col].fillna(df_dup3[col].mode()[0])

df_dup3.head()

,Category,Price,Rating,Stock,Discount,Rating_missing,Discount_missing,Category_missing,Stock_missing
0,C,5548.0,1.870322,In Stock,0.0,0,0,1,1
1,C,3045.0,4.757798,In Stock,38.0,0,0,1,1
2,C,4004.0,3.038293,In Stock,0.0,1,0,1,0
3,C,4808.0,1.492085,In Stock,33.0,0,0,1,1
4,C,1817.0,3.038293,Out of Stock,23.0,1,0,1,0


In [212]:
df_dup1 = df_dup1.dropna(subset=['Price'])
df_dup2 = df_dup2.dropna(subset=['Price'])
df_dup3 = df_dup3.dropna(subset=['Price'])

## Testing

In [213]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import pandas as pd

def evaluate_dataset(df, target="Price"):
    X = df.drop(columns=[target])
    y = df[target]

    X = pd.get_dummies(X, drop_first=True)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42
    )

    model = RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    return {
        "R2": r2_score(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred)
    }

In [214]:
results = {}

results["df_dup1"] = evaluate_dataset(df_dup1)
results["df_dup2"] = evaluate_dataset(df_dup2)
results["df_dup3"] = evaluate_dataset(df_dup3)

results

{'df_dup1': {'R2': -0.13931175208326074, 'MAE': 2527.4737853290158},
 'df_dup2': {'R2': -0.14107467005227603, 'MAE': 2533.0555995180507},
 'df_dup3': {'R2': -0.1498513095533689, 'MAE': 2548.233408417108}}

In [215]:
df_dup1.equals(df_dup2)
df_dup1.equals(df_dup3)

False

### Conclusion:

Different missing value strategies were evaluated using a consistent regression model. This experiment demonstrates how preprocessing choices can directly impact model performance, even before feature engineering or model tuning.